# 📅 Exercício Extra 4: Estacionamento Autónomo por AprilTags 🅿️🤖

Nas fábricas modernas e nos armazéns inteligentes (como os da Amazon), os robôs movem-se e estacionam de forma autónoma seguindo marcadores visuais colados nas paredes ou no chão. Estes marcadores chamam-se **AprilTags**.

Neste exercício, vamos programar o JetRacer para encontrar a **Tag com o ID 0** (que representa a nossa vaga de estacionamento), alinhar-se com ela e avançar até parar perfeitamente estacionado.

### 📐 A Lógica de Condução e Estacionamento
* **Direção (`steering`):** Calculamos o erro entre o centro do ecrã (`150`) e o centro da Tag. O robô vai virar as rodas para corrigir esse desvio.
* **Aproximação (`throttle`):** O robô mede o tamanho (largura) da Tag em píxeis. Se a Tag for pequena, significa que o estacionamento está longe e o robô avança. Quando a Tag atingir uma largura superior a **100 píxeis**, significa que chegámos ao destino e o robô trava!

---

### 🛠️ Instruções Passo a Passo
1. ⚠️ **Segurança:** Faz os primeiros testes com as **rodas no ar** (em cima do bloco)!
2. Se ainda não o fizeste, instala a biblioteca de deteção correndo a célula: `!pip3 install pupil-apriltags`.
3. Mostra uma imagem da **AprilTag da família 36h11 (ID 0)** à câmara (podes usar o ecrã do telemóvel).
4. Executa o código e observa se as rodas reagem corretamente para alinhar e parar o veículo.

In [1]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jethelper import mover, parar
import time

# 1. Inicializar o Detetor Industrial de AprilTags
try:
    from pupil_apriltags import Detector
    detetor = Detector(families='tag36h11')
    print("--- PILOTO DE ESTACIONAMENTO AUTÓNOMO ATIVO ---")
except ImportError:
    print("❌ Erro: Instala a biblioteca correndo primeiro: !pip3 install pupil-apriltags")

# Parâmetros de afinação de condução
K_direcao = 0.006          # Força da correção de direção para alinhar com a Tag
velocidade_aproximacao = 0.12 # Velocidade suave de aproximação (12%)

# 2. Inicializar a câmara e criar a interface visual
camera = CSICamera(width=300, height=300, capture_width=1280, capture_height=720, capture_fps=15)
imagem_widget = widgets.Image(format='jpeg', width=300, height=300)
botao_desligar = widgets.Button(description="❌ DESLIGAR SISTEMA", button_style='danger')

display(imagem_widget, botao_desligar)
sistema_ativo = True

# 3. Função de Navegação Assíncrona
def conduzir_ate_vaga(change):
    global sistema_ativo
    if not sistema_ativo:
        parar()
        return
        
    frame = change['new']
    cinzento = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Procura marcadores digitais no frame
    tags = detetor.detect(cinzento)
    
    centro_imagem_x = 150 # Centro horizontal do ecrã (300/2)
    
    if len(tags) == 0:
        print("A procurar vaga de estacionamento (Tag ID 0)... 🔍             ", end='\r')
        parar() # Se não vir nenhuma tag, fica imóvel por segurança
    else:
        for tag in tags:
            # Ignora tags que não sejam o nosso lugar de estacionamento (ID 0)
            if tag.tag_id != 0:
                continue
                
            # Obter coordenadas dos cantos da tag
            (ptA, ptB, ptC, ptD) = tag.corners
            ptB = (int(ptB[0]), int(ptB[1]))
            ptD = (int(ptD[0]), int(ptD[1]))
            
            # Desenha o quadrado de confirmação verde à volta da tag
            cv2.rectangle(frame, ptB, ptD, (0, 255, 0), 2)
            
            # Calcular a largura atual da tag (Mede a distância ao alvo)
            largura_tag = abs(ptD[0] - ptB[0])
            
            # Calcular o centro horizontal da tag
            centro_tag_x = int(tag.center[0])
            
            # Calcular o desvio de alinhamento
            erro_x = centro_tag_x - centro_imagem_x
            
            # --- 🎯 LÓGICA DE MOVIMENTO E ESTACIONAMENTO ---
            if largura_tag > 105:
                # CASO A: Chegámos ao lugar de estacionamento! (Tag muito grande no ecrã)
                cv2.putText(frame, "🅿️ ESTACIONADO COM SUCESSO!", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
                parar()
                print(f"🎉 Destino Atingido! Largura da Tag: {largura_tag} px -> Motores Desligados!", end='\r')
            else:
                # CASO B: A vaga está livre mas ainda longe. Avança e corrige a direção
                direcao = erro_x * K_direcao
                direcao = max(min(direcao, 1.0), -1.0) # Proteção de limites
                
                mover(velocidade_aproximacao, direcao)
                cv2.putText(frame, "A estacionar...", (int(ptB[0]), int(ptB[1]) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                print(f"🚀 Alinhando | Erro X: {erro_x:3d} px | Largura Tag: {largura_tag:3d} px/105px", end='\r')
                
    # Exibe o resultado no Jupyter
    _, jpeg = cv2.imencode('.jpg', frame)
    imagem_widget.value = jpeg.tobytes()
    time.sleep(0.02)

# Ativar a escuta da câmara
camera.observe(conduzir_ate_vaga, names='value')

# 4. Botão de Encerramento e Segurança
def desligar_tudo(b):
    global sistema_ativo
    sistema_ativo = False
    print("\n[A desativar piloto de estacionamento...]")
    parar()
    try:
        camera.unobserve(conduzir_ate_vaga, names='value')
        camera.running = False
    except:
        pass
    botao_desligar.description = "🛑 ESTACIONAMENTO INATIVO"
    botao_desligar.button_style = "info"
    botao_desligar.disabled = True
    print("Sistema imobilizado com segurança!")

botao_desligar.on_click(desligar_tudo)
camera.running = True

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


--- PILOTO DE ESTACIONAMENTO AUTÓNOMO ATIVO ---


Image(value=b'', format='jpeg', height='300', width='300')

Button(button_style='danger', description='❌ DESLIGAR SISTEMA', style=ButtonStyle())